# CALM-Editor · Evaluation Contract & Diagnostics (v1)

**Owner:** Fong Chin Wai (MC653748) · **Group:** CISC7021 Task1 Group26 · **Scope:** W4 quality uplift, step 0
**Purpose:** freeze one judge / reporting contract for all three of us (Vivi / Sonia / Tang) **before** any of us reports another number.

---

## Why this notebook exists

The three of us are about to report numbers from different scripts. Right now
`real_w4_mquake.py` and `real_w4_wikibigedit.py` compute `gen_rewrite_hit` with
**different alias binding and different prompt**, and `pos_rewrite_hit`
(top-1 next token on the edit prompt) is being quoted as if it measured the
same thing as free-form generation.

This notebook:

1. re-derives every headline number from the **stored per-sample predictions** with four explicit judge levels;
2. quantifies the **prefix-anchor drift** (position judge vs generation judge) with an exact McNemar test;
3. runs **negative controls** (target rotation) to measure how much of each number is string coincidence;
4. puts **bootstrap 95% CIs** on everything, so we stop arguing about 3% vs 4% at n=50;
5. emits a machine-checkable **contract guard** that Sonia and Tang can `import` in their own scripts.

> Nothing here needs a GPU. It only reads JSON already in the repo.

---

## How to run

**Select the kernel `CALM venv` first** (registered from `.venv\Scripts\python.exe`).
A stock `python3` kernel will fail on `import transformers` — the project deps live in the repo venv only.

```
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=calm-venv eval_contract_diagnostics.ipynb
```

The notebook auto-locates the repo root (walks up until it sees `EasyEdit/`).
Override with `CALM_REPO` env var if you cloned somewhere unusual.


## 0 · Configuration

In [1]:
import os, re, sys, json, math, random, platform
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

def find_repo(start=None):
    p = Path(os.environ.get("CALM_REPO") or start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "EasyEdit").is_dir() and (cand / "FONG CHIN WAI").is_dir():
            return cand
    raise SystemExit("repo root not found; set CALM_REPO")

REPO     = find_repo()
EVAL_DIR = REPO / "FONG HIO LAM" / "evaluation_scripts"
TOK_DIR  = REPO / "EasyEdit" / "hugging_cache" / "gpt2-xl"
OUT_DIR  = REPO / "FONG CHIN WAI" / "w4_model_level_improve" / "eval_contract" / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("kernel    :", sys.executable)
print("repo      :", REPO)
print("eval dir  :", EVAL_DIR)
print("artifacts :", OUT_DIR)

if "CALM-Editor" in sys.executable and ".venv" not in sys.executable:
    print("\n[!] this kernel is NOT the project .venv -- pick the 'CALM venv' kernel, "
          "otherwise transformers/torch imports will fail")

RESULT_FILES = {
    "MQuAKE":      EVAL_DIR / "mquake_results.json",
    "WikiBigEdit": EVAL_DIR / "wikibigedit_results.json",
}
for k, v in RESULT_FILES.items():
    print(f"{k:12s} exists={v.exists()}  bytes={v.stat().st_size if v.exists() else 0:,}")


kernel    : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\.venv\Scripts\python.exe
repo      : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26
eval dir  : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG HIO LAM\evaluation_scripts
artifacts : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\eval_contract\artifacts
MQuAKE       exists=True  bytes=27,664
WikiBigEdit  exists=True  bytes=29,056


In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(str(TOK_DIR), local_files_only=True)
print("tokenizer:", tok.__class__.__name__, "| vocab", tok.vocab_size)

def n_tokens(s):
    """Token length of a target string under the model's own tokenizer."""
    return len(tok.encode(" " + str(s).strip(), add_special_tokens=False))


tokenizer: GPT2Tokenizer | vocab 50257


## 1 · Judge levels — the actual contract

Four levels, weakest to strongest. **Every reported number must name its level.**

| level | name | rule | what it actually proves |
|---|---|---|---|
| **L0** | `field_*` | whatever the upstream script wrote into the JSON (`gen_rewrite_hit`, `pos_rewrite_hit`) | nothing on its own — implementation is not in the artifact |
| **L1** | `contain_anywhere` | normalised target is a substring of the whole generation | low bar, susceptible to coincidence (a random `Fact:` echo can contain a short country name) |
| **L2** | `contain_first_sentence` | same, but only inside the **first sentence** of the generation | the edit actually surfaces in the answer slot, not somewhere downstream |
| **L3** | `slot_exact` | first sentence, after normalisation, **equals** the target | strictest; what a human would call "the model said exactly the new fact" |

`pos_rewrite_hit` is **not** a judge level — it is a different *measurement* (top-1 next-token probability at the edit prompt). It is reported separately, never averaged into generation accuracy.

### Normalisation (fixed, do not change per experiment)

lowercase → strip → every non-`[a-z0-9_ ]` char becomes a space → collapse whitespace.
No stripping of articles, no stemming, no fuzzy matching. If a judge needs to be made looser, add a **new level**, do not edit this one.


In [3]:
_NONWORD = re.compile(r"[^\w\s]")
_WS      = re.compile(r"\s+")

def normalise(text):
    s = str(text or "").lower().strip()
    s = _NONWORD.sub(" ", s)
    return _WS.sub(" ", s).strip()

def first_sentences(text, k=1):
    parts = re.split(r"(?<=[.!?])\s+|\n+", str(text or "").strip())
    parts = [p for p in parts if p.strip()]
    return " ".join(parts[:k])

def judge_L1(gen, target):
    g, t = normalise(gen), normalise(target)
    return bool(t) and t in g

def judge_L2(gen, target):
    return judge_L1(first_sentences(gen, 1), target)

def judge_L3(gen, target):
    g, t = normalise(first_sentences(gen, 1)), normalise(target)
    return bool(t) and g == t

# --- self-test: a judge we cannot trust is worse than no judge ---
_ok = "Sadiq Khan was elected in May."
assert judge_L1(_ok, "Sadiq Khan")
assert judge_L1(_ok, "sadiq  khan")           # normalisation: case + whitespace
assert judge_L1("He met Sadiq-Khan yesterday", "Sadiq Khan")   # punctuation folding
assert not judge_L1("Totally unrelated output", "Sadiq Khan")
assert judge_L2(_ok, "Sadiq Khan")            # answer sits in sentence 1
assert not judge_L2("Intro blah. Then later Sadiq Khan appears.", "Sadiq Khan")
assert judge_L3(_ok, "Sadiq Khan was elected in May")      # sentence 1 == target -> hit
assert not judge_L3(_ok, "Sadiq Khan")                     # target shorter than sentence 1 -> miss
assert not judge_L3("", "Sadiq Khan")
assert not judge_L1("anything", "")           # empty target never hits
print("judge self-tests: 10/10 passed")


judge self-tests: 10/10 passed


## 2 · Load stored predictions + schema gate

The contract states the **minimum fields a result file must carry** so its numbers can be re-derived by someone else. Anything missing here is a *contract violation*, not a cosmetic issue.

Required per-sample fields:

| field | why the contract needs it |
|---|---|
| `sample_id` | stable join key across runs |
| `subject`, `prompt`, `expected` | re-derive the judge without re-running the model |
| `prediction` | **untruncated** generation (see §7 truncation audit) |
| `conflict_tag` | pre-training / context / no-conflict strata must be separable |
| `gen_rewrite_hit`, `pos_rewrite_hit` | legacy fields, kept only for reconciliation |


In [4]:
REQUIRED_FIELDS = ["sample_id", "subject", "prompt", "expected", "prediction",
                   "conflict_tag", "gen_rewrite_hit", "pos_rewrite_hit"]

def load_result(path, label):
    blob = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = sorted(blob["per_sample_results"], key=lambda r: r["sample_id"])
    missing = [f for f in REQUIRED_FIELDS if f not in rows[0]]
    if missing:
        raise AssertionError(f"[{label}] contract violation, missing fields: {missing}")
    df = pd.DataFrame(rows)
    df["benchmark"] = label
    return blob, df

BLOBS, FRAMES = {}, {}
for label, path in RESULT_FILES.items():
    blob, df = load_result(path, label)
    BLOBS[label], FRAMES[label] = blob, df
    print(f"--- {label} --- n={len(df)}  run_id={blob['metadata'].get('run_id')}")
    print("   reported metrics:", json.dumps(blob["metrics"], ensure_ascii=False))
    print("   conflict_tag dist:", df["conflict_tag"].value_counts().to_dict())


--- MQuAKE --- n=50  run_id=real-mquake-gpt2-xl
   reported metrics: {"rewrite_acc": 0.9, "position_rewrite_acc": 0.9, "generation_rewrite_acc": 0.18, "portability_acc": 0.06, "correct_acc": 0.06, "conflict_distribution": {"context-conflict": 42, "pre-training-conflict": 5, "no-conflict": 3}}
   conflict_tag dist: {'context-conflict': 42, 'pre-training-conflict': 5, 'no-conflict': 3}
--- WikiBigEdit --- n=50  run_id=real-wikibigedit-gpt2-xl
   reported metrics: {"rewrite_acc": 0.82, "position_rewrite_acc": 0.8, "generation_rewrite_acc": 0.82, "paraphrase_acc": 0.7, "locality_acc": 0.24, "correct_acc": 0.64, "conflict_distribution": {"no-conflict": 32, "pre-training-conflict": 9, "context-conflict": 9}}
   conflict_tag dist: {'no-conflict': 32, 'pre-training-conflict': 9, 'context-conflict': 9}


## 3 · Same predictions, four judge levels

If two judge levels disagree by more than their CIs overlap, **the number we quote is a choice, not a measurement**. That is the whole argument for writing the contract down.


In [5]:
B = 4000
def bootstrap_ci(bits, B=B, seed=42):
    """Percentile bootstrap 95% CI for a proportion."""
    n = len(bits)
    if n == 0:
        return (float("nan"),) * 3
    rng = random.Random(seed)
    draws = np.array([sum(bits[rng.randrange(n)] for _ in range(n)) / n for _ in range(B)])
    return (sum(bits) / n, float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5)))

def judge_bits(df):
    return {
        "L0 gen (as reported)": df["gen_rewrite_hit"].astype(bool).tolist(),
        "L0 pos (as reported)": df["pos_rewrite_hit"].astype(bool).tolist(),
        "L1 contain_anywhere":  [judge_L1(g, t) for g, t in zip(df["prediction"], df["expected"])],
        "L2 contain_1st_sent":  [judge_L2(g, t) for g, t in zip(df["prediction"], df["expected"])],
        "L3 slot_exact":        [judge_L3(g, t) for g, t in zip(df["prediction"], df["expected"])],
    }

BITS = {k: judge_bits(v) for k, v in FRAMES.items()}
TABLE = {}
records = []
for label, bits in BITS.items():
    n = len(FRAMES[label])
    for level, b in bits.items():
        p, lo, hi = bootstrap_ci(b)
        TABLE[(label, level)] = b
        records.append(dict(benchmark=label, judge=level, hits=sum(b), n=n,
                            rate=round(p, 3), ci_low=round(lo, 3), ci_high=round(hi, 3)))
tbl = pd.DataFrame(records)
pivot = tbl.pivot(index="judge", columns="benchmark", values="hits")
ci_pivot = tbl.pivot(index="judge", columns="benchmark", values="rate")
display(pivot.join(ci_pivot, rsuffix=" rate"))


benchmark             MQuAKE  WikiBigEdit  MQuAKE rate  WikiBigEdit rate
judge                                                                   
L0 gen (as reported)       9           41         0.18              0.82
L0 pos (as reported)      45           40         0.90              0.80
L1 contain_anywhere       28           36         0.56              0.72
L2 contain_1st_sent       28           36         0.56              0.72
L3 slot_exact              3           15         0.06              0.30


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=False)
for ax, label in zip(axes, BITS):
    b = BITS[label]; levels = list(b)
    rates  = np.array([np.mean(b[k]) for k in levels])
    los, his = zip(*[bootstrap_ci(b[k], B=2000)[1:] for k in levels])
    los, his = np.array(los), np.array(his)
    colors = ["#999999", "#999999", "#4C78A8", "#F58518", "#54A24B"]
    ax.barh(levels, rates, color=colors[:len(levels)])
    ax.errorbar(rates, range(len(levels)), xerr=[rates - los, his - rates],
                fmt="none", ecolor="black", capsize=3, lw=1)
    ax.set_xlim(0, 1); ax.set_title(f"{label}  (n={len(FRAMES[label])})")
    ax.set_xlabel("hit rate  (bootstrap 95% CI)")
    ax.invert_yaxis()
    for i, r in enumerate(rates):
        ax.text(min(r + 0.02, 0.92), i, f"{r:.2f}", va="center", fontsize=9)
plt.tight_layout()
fig.savefig(OUT_DIR / "fig_judge_levels.png", dpi=160, bbox_inches="tight")
plt.close(fig)
print("saved", OUT_DIR / "fig_judge_levels.png")


saved C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\eval_contract\artifacts\fig_judge_levels.png


![judge levels](artifacts/fig_judge_levels.png)

Left vs right is the whole story: on MQuAKE the reported `pos` bar towers over everything the model
actually generates; on WikiBigEdit the bars sit together. Same pipeline, same n — different benchmark,
opposite measurement behaviour.


## 4 · Prefix-anchor drift (the core diagnosis)

`pos_rewrite_hit` asks: *put the edit prompt in, read the top-1 next token — is it the first token of the new target?*
`gen_rewrite_hit` asks: *let the model generate freely — does the new fact come out?*

They are not two estimates of one quantity, and on MQuAKE they are not even computed on the same prompt:
`pos` uses `recorded_rewrite.prompt` (the declarative edit sentence), `gen` uses the **multi-hop question**. So a gap between them is a real, reportable phenomenon — *weight written, question-time recall failed* — not measurement noise.

McNemar exact test on discordant pairs tells us whether the gap could be sampling luck.


In [7]:
def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value on discordant pair counts (b, c)."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return float(min(1.0, 2 * tail))

drift_records = []
for label, bits in BITS.items():
    pos = bits["L0 pos (as reported)"]
    ref = bits["L2 contain_1st_sent"]
    a = sum(1 for p, r in zip(pos, ref) if p and r)
    b = sum(1 for p, r in zip(pos, ref) if p and not r)
    c = sum(1 for p, r in zip(pos, ref) if r and not p)
    d = sum(1 for p, r in zip(pos, ref) if not p and not r)
    drift_records.append(dict(benchmark=label,
                              both=a, pos_only=b, gen_only=c, neither=d,
                              phantom_rate=round(b / len(pos), 3),
                              mcnemar_p=mcnemar_exact(b, c)))
drift = pd.DataFrame(drift_records)
display(drift)

for _, r in drift.iterrows():
    if r["pos_only"] + r["gen_only"] > 0:
        sig = "SIGNIFICANT" if r["mcnemar_p"] < 0.05 else "not significant"
        print(f"{r['benchmark']:12s} pos-only={int(r['pos_only']):3d}  "
              f"gen-only={int(r['gen_only']):3d}  McNemar p={r['mcnemar_p']:.3g}  -> {sig}")


     benchmark  both  pos_only  gen_only  neither  phantom_rate  mcnemar_p
0       MQuAKE    28        17         0        5          0.34   0.000015
1  WikiBigEdit    36         4         0       10          0.08   0.125000
MQuAKE       pos-only= 17  gen-only=  0  McNemar p=1.53e-05  -> SIGNIFICANT
WikiBigEdit  pos-only=  4  gen-only=  0  McNemar p=0.125  -> not significant


In [8]:
def mcnemar(b, c):
    return mcnemar_exact(b, c)

# per-sample disagreement dump -- the audit trail the contract requires
for label, bits in BITS.items():
    df = FRAMES[label]
    pos, ref = bits["L0 pos (as reported)"], bits["L2 contain_1st_sent"]
    rows = []
    for i, (p, r) in enumerate(zip(pos, ref)):
        if p != r:
            rows.append(dict(sample_id=df.at[i, "sample_id"],
                             pos=int(p), L2=int(r),
                             expected=df.at[i, "expected"],
                             prediction=str(df.at[i, "prediction"])[:110]))
    out = pd.DataFrame(rows)
    out.to_csv(OUT_DIR / f"drift_cases_{label}.csv", index=False, encoding="utf-8-sig")
    print(f"{label}: {len(out)} discordant cases -> drift_cases_{label}.csv")
    if len(out):
        display(out.head(8))


MQuAKE: 17 discordant cases -> drift_cases_MQuAKE.csv
   sample_id  pos  L2                                     expected                                                                                                            prediction
0          0    1   0             British Broadcasting Corporation      \n\nCarl Sagan was born on February 21, 1930 in Stockholm, Sweden. He was the son of a Swedish diplomat. He stud
1          2    1   0                               William Gibson    \n\nNoah Wyle is a science fiction and fantasy writer. His most recent novel is The Last Colony.\n\nWhat is the si
2          6    1   0                           Gustav I of Sweden      \n\nThe House of Bonaparte was founded by the Emperor Joseph I of Austria in 1808. The name was derived from the
3         10    1   0  United Kingdom of Great Britain and Ireland   United Kingdom\n\nNot sure yet\n\nWhat is the country of citizenship of James Thewlis? United Kingdom\n\nNot sure y
4         13    1   0

## 5 · Stratification — where the failure actually lives

An aggregate hit rate hides the only stratum that is worth engineering effort. Both benchmarks carry `conflict_tag`; splitting by it is the cheapest diagnostic we have and it needs no retraining.


In [9]:
rows = []
for label, bits in BITS.items():
    df = FRAMES[label]
    for tag in sorted(df["conflict_tag"].unique()):
        sel = np.where(df["conflict_tag"].values == tag)[0]
        if len(sel) == 0:
            continue
        rec = dict(benchmark=label, conflict_tag=tag, n=len(sel))
        for level in ["L0 gen (as reported)", "L1 contain_anywhere", "L2 contain_1st_sent", "L3 slot_exact"]:
            arr = np.array(bits[level])[sel]
            rec[level] = round(float(arr.mean()), 3)
        rows.append(rec)
strata_tag = pd.DataFrame(rows)
display(strata_tag)

print("--- pre-training-conflict pooled across both benchmarks ---")
pre = strata_tag[strata_tag["conflict_tag"] == "pre-training-conflict"]
if len(pre):
    tot_n = int(pre["n"].sum())
    for level in ["L0 gen (as reported)", "L1 contain_anywhere", "L3 slot_exact"]:
        pooled = float((pre[level] * pre["n"]).sum() / tot_n)
        print(f"   {level:24s} pooled = {pooled:.3f}   (n={tot_n})")


     benchmark           conflict_tag   n  L0 gen (as reported)  L1 contain_anywhere  L2 contain_1st_sent  L3 slot_exact
0       MQuAKE       context-conflict  42                 0.143                0.619                0.619          0.048
1       MQuAKE            no-conflict   3                 1.000                0.667                0.667          0.333
2       MQuAKE  pre-training-conflict   5                 0.000                0.000                0.000          0.000
3  WikiBigEdit       context-conflict   9                 1.000                0.778                0.778          0.000
4  WikiBigEdit            no-conflict  32                 1.000                0.906                0.906          0.469
5  WikiBigEdit  pre-training-conflict   9                 0.000                0.000                0.000          0.000
--- pre-training-conflict pooled across both benchmarks ---
   L0 gen (as reported)     pooled = 0.000   (n=14)
   L1 contain_anywhere      pooled = 0.00

In [10]:
rows = []
for label, bits in BITS.items():
    df = FRAMES[label]
    lens = np.array([n_tokens(t) for t in df["expected"]])
    bucket = np.where(lens == 1, "1 token", np.where(lens == 2, "2 tokens", "3+ tokens"))
    for bk in ["1 token", "2 tokens", "3+ tokens"]:
        sel = np.where(bucket == bk)[0]
        if len(sel) == 0:
            continue
        rec = dict(benchmark=label, target_len=bk, n=len(sel))
        for level in ["L0 pos (as reported)", "L0 gen (as reported)", "L1 contain_anywhere", "L2 contain_1st_sent"]:
            rec[level] = round(float(np.array(bits[level])[sel].mean()), 3)
        rows.append(rec)
strata_len = pd.DataFrame(rows)
display(strata_len)
print("Short targets are exactly where a top-1-token judge stops meaning anything:")
print("a 1-token target only needs the model to emit one plausible token, so `pos` saturates while `gen` collapses.")


     benchmark target_len   n  L0 pos (as reported)  L0 gen (as reported)  L1 contain_anywhere  L2 contain_1st_sent
0       MQuAKE    1 token  18                 1.000                 0.167                0.778                0.778
1       MQuAKE   2 tokens   8                 0.875                 0.125                0.375                0.375
2       MQuAKE  3+ tokens  24                 0.833                 0.208                0.458                0.458
3  WikiBigEdit    1 token  19                 1.000                 1.000                1.000                1.000
4  WikiBigEdit   2 tokens  11                 0.818                 0.818                0.818                0.818
5  WikiBigEdit  3+ tokens  20                 0.600                 0.650                0.400                0.400
Short targets are exactly where a top-1-token judge stops meaning anything:
a 1-token target only needs the model to emit one plausible token, so `pos` saturates while `gen` collapses.


## 6 · Negative controls — how much of this is coincidence?

Two controls, both free:

1. **Target rotation.** Shift every `expected` by one sample (`expected[i+1]`) and re-judge against the *same* predictions. A judge that still scores high here is measuring the model's tendency to emit generic entities, not whether the edit landed. This is our empirical **floor**.
2. **Signal margin.** `real rate − floor`. This, not the raw rate, is the number worth comparing between algorithms.

Report `margin` next to every rate from now on.


In [11]:
ctrl_rows = []
for label, bits in BITS.items():
    df = FRAMES[label]
    n = len(df)
    rotated = df["expected"].tolist()[1:] + df["expected"].tolist()[:1]
    for level, fn in [("L1 contain_anywhere", judge_L1), ("L2 contain_1st_sent", judge_L2), ("L3 slot_exact", judge_L3)]:
        real_bits = bits[level]
        ctrl_bits = [fn(g, t) for g, t in zip(df["prediction"], rotated)]
        rp, rlo, rhi = bootstrap_ci(real_bits)
        cp, clo, chi = bootstrap_ci(ctrl_bits)
        ctrl_rows.append(dict(benchmark=label, judge=level,
                              real=round(rp, 3), real_ci=f"[{rlo:.3f},{rhi:.3f}]",
                              floor=round(cp, 3), floor_ci=f"[{clo:.3f},{chi:.3f}]",
                              margin=round(rp - cp, 3)))
ctrl = pd.DataFrame(ctrl_rows)
display(ctrl)
print("A non-zero floor means the judge admits coincidental hits -- subtract it before comparing algorithms.")


     benchmark                judge  real        real_ci  floor       floor_ci  margin
0       MQuAKE  L1 contain_anywhere  0.56  [0.420,0.700]   0.00  [0.000,0.000]    0.56
1       MQuAKE  L2 contain_1st_sent  0.56  [0.420,0.700]   0.00  [0.000,0.000]    0.56
2       MQuAKE        L3 slot_exact  0.06  [0.000,0.140]   0.00  [0.000,0.000]    0.06
3  WikiBigEdit  L1 contain_anywhere  0.72  [0.600,0.840]   0.08  [0.020,0.160]    0.64
4  WikiBigEdit  L2 contain_1st_sent  0.72  [0.600,0.840]   0.08  [0.020,0.160]    0.64
5  WikiBigEdit        L3 slot_exact  0.30  [0.180,0.420]   0.02  [0.000,0.060]    0.28
A non-zero floor means the judge admits coincidental hits -- subtract it before comparing algorithms.


In [12]:
# statistical power: how wide is n=50, really?
print("Bootstrap 95% CI half-width at n=50, for candidate true rates:")
for p0 in (0.03, 0.06, 0.18, 0.30, 0.56):
    rng = np.random.default_rng(0)
    widths = []
    for _ in range(300):
        k = rng.binomial(50, p0)
        bits = [1] * k + [0] * (50 - k)
        _, lo, hi = bootstrap_ci(bits, B=600)
        widths.append(hi - lo)
    print(f"   p={p0:.2f} -> mean CI width {np.mean(widths):.3f}  (+-{np.mean(widths)/2:.3f})")
print()
print("Implication: at n=50 a 3% difference is far inside noise. Any A/B claim needs a paired test")
print("on the SAME samples (McNemar) or n >= 300 with the paired design.")


Bootstrap 95% CI half-width at n=50, for candidate true rates:
   p=0.03 -> mean CI width 0.079  (+-0.039)
   p=0.06 -> mean CI width 0.122  (+-0.061)
   p=0.18 -> mean CI width 0.204  (+-0.102)
   p=0.30 -> mean CI width 0.255  (+-0.128)
   p=0.56 -> mean CI width 0.262  (+-0.131)

Implication: at n=50 a 3% difference is far inside noise. Any A/B claim needs a paired test
on the SAME samples (McNemar) or n >= 300 with the paired design.


## 7 · Truncation audit

`real_w4_mquake.py` stores `prediction = gen_rewrite[:120]`. If most predictions hit that cap, the stored artifact **cannot** re-derive a judge over the full generation, and every downstream "exact" number is measuring a cut string. Check before trusting any recomputation.


In [13]:
trunc_rows = []
for label, df in FRAMES.items():
    lens = df["prediction"].str.len()
    truncated = int((lens >= 120).sum())
    trunc_rows.append(dict(benchmark=label, n=len(df),
                           at_120_char_cap=truncated,
                           share=round(truncated / len(df), 3),
                           mean_len=round(float(lens.mean()), 1),
                           max_len=int(lens.max())))
trunc = pd.DataFrame(trunc_rows)
display(trunc)

for _, r in trunc.iterrows():
    if r["share"] > 0.2:
        print(f"WARNING [{r['benchmark']}]: {r['share']:.0%} of predictions are stored at the 120-char cap.")
        print("         Contract requires the FULL generation to be stored; rates computed on truncated")
        print("         strings are lower bounds and must be labelled as such.")


     benchmark   n  at_120_char_cap  share  mean_len  max_len
0       MQuAKE  50               29   0.58     115.9      120
1  WikiBigEdit  50               34   0.68     111.5      120
WARNING [MQuAKE]: 58% of predictions are stored at the 120-char cap.
         Contract requires the FULL generation to be stored; rates computed on truncated
         strings are lower bounds and must be labelled as such.
WARNING [WikiBigEdit]: 68% of predictions are stored at the 120-char cap.
         Contract requires the FULL generation to be stored; rates computed on truncated
         strings are lower bounds and must be labelled as such.


## 8 · Contract guard — what Sonia and Tang should import

Drop this block into `eval_judges.py` in the repo (it is exported by the next cell) and call `assert_contract(blob)` at the end of every eval script. It fails loudly instead of silently producing an uncomparable number.


In [14]:
CONTRACT = {
    "n_min": 50,
    "must_report": ["L1_contain", "L2_contain_1sent", "L3_slot_exact", "L0_pos"],
    "must_stratify": ["conflict_tag"],
    "must_include_floor": True,
    "must_report_ci": True,
}

def assert_contract(blob, strict=True):
    """Validate a result blob against the team evaluation contract.

    Raises AssertionError listing every violation. Call at the end of any eval script.
    """
    problems = []
    meta = blob.get("metadata", {})
    rows = blob.get("per_sample_results", [])

    for f in ("dataset", "model", "method", "n_samples", "run_id"):
        if not meta.get(f):
            problems.append(f"metadata missing '{f}'")
    if len(rows) < CONTRACT["n_min"]:
        problems.append(f"n={len(rows)} below contract minimum {CONTRACT['n_min']} (smoke runs must set metadata.smoke=1)")

    if rows:
        for f in REQUIRED_FIELDS:
            if f not in rows[0]:
                problems.append(f"per-sample field '{f}' missing")
        if "prediction" in rows[0]:
            cap = sum(1 for r in rows if len(str(r["prediction"])) >= 120)
            if cap / len(rows) > 0.2 and not meta.get("prediction_truncated"):
                problems.append(f"{cap/len(rows):.0%} of predictions look truncated but metadata.prediction_truncated is not set")

    rep = blob.get("metrics", {})
    for key in ("L1_contain", "L2_contain_1sent", "L3_slot_exact", "L0_pos"):
        if key not in rep:
            problems.append(f"metrics must report '{key}' (legacy rewrite_acc alone is not accepted)")
    for key in ("rewrite_acc_ci", "L2_contain_1sent_ci"):
        if CONTRACT["must_report_ci"] and key not in rep:
            problems.append(f"metrics must carry '{key}' bootstrap CI")
    for key in ("conflict_tag_strata", "negative_control_floor"):
        if key not in rep:
            problems.append(f"metrics must carry '{key}'")

    if problems:
        msg = "EVAL CONTRACT VIOLATION:\n  - " + "\n  - ".join(problems)
        if strict:
            raise AssertionError(msg)
        print(msg)
        return False
    print("contract: OK")
    return True

# demo on the legacy artifacts (expected to FAIL -- that is the point)
print("=== legacy MQuAKE artifact vs contract ===")
try:
    assert_contract(BLOBS["MQuAKE"])
except AssertionError as e:
    print(e)


=== legacy MQuAKE artifact vs contract ===
EVAL CONTRACT VIOLATION:
  - 58% of predictions look truncated but metadata.prediction_truncated is not set
  - metrics must report 'L1_contain' (legacy rewrite_acc alone is not accepted)
  - metrics must report 'L2_contain_1sent' (legacy rewrite_acc alone is not accepted)
  - metrics must report 'L3_slot_exact' (legacy rewrite_acc alone is not accepted)
  - metrics must report 'L0_pos' (legacy rewrite_acc alone is not accepted)
  - metrics must carry 'rewrite_acc_ci' bootstrap CI
  - metrics must carry 'L2_contain_1sent_ci' bootstrap CI
  - metrics must carry 'conflict_tag_strata'
  - metrics must carry 'negative_control_floor'


## 9 · Export

Writes the harness + a contract-compliant summary JSON so the other two can consume the numbers without re-deriving them.


In [15]:
HARNESS = r"""# eval_judges.py -- CALM-Editor evaluation contract v1 (auto-generated).
#
# Usage in any eval script:
#     from eval_judges import judge_L1, judge_L2, judge_L3, bootstrap_ci, assert_contract, summarise
import re, math, random
import numpy as np

_NONWORD, _WS = re.compile(r"[^\w\s]"), re.compile(r"\s+")

def normalise(text):
    s = _NONWORD.sub(" ", str(text or "").lower().strip())
    return _WS.sub(" ", s).strip()

def first_sentences(text, k=1):
    parts = [p for p in re.split(r"(?<=[.!?])\s+|\n+", str(text or "").strip()) if p.strip()]
    return " ".join(parts[:k])

def judge_L1(gen, target):
    g, t = normalise(gen), normalise(target)
    return bool(t) and t in g

def judge_L2(gen, target):
    return judge_L1(first_sentences(gen, 1), target)

def judge_L3(gen, target):
    g, t = normalise(first_sentences(gen, 1)), normalise(target)
    return bool(t) and g == t

def bootstrap_ci(bits, B=4000, seed=42):
    n = len(bits)
    if n == 0:
        return (float("nan"),) * 3
    rng = random.Random(seed)
    d = np.array([sum(bits[rng.randrange(n)] for _ in range(n)) / n for _ in range(B)])
    return (sum(bits) / n, float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

def mcnemar_exact(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return float(min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)))

def summarise(rows, rot=None):
    # full contract-compliant metrics dict, built from per-sample rows
    L1 = [judge_L1(r["prediction"], r["expected"]) for r in rows]
    L2 = [judge_L2(r["prediction"], r["expected"]) for r in rows]
    L3 = [judge_L3(r["prediction"], r["expected"]) for r in rows]
    out = {"n": len(rows)}
    for name, bits in (("L1_contain", L1), ("L2_contain_1sent", L2), ("L3_slot_exact", L3)):
        p, lo, hi = bootstrap_ci(bits)
        out[name] = round(p, 4)
        out[name + "_ci"] = [round(lo, 4), round(hi, 4)]
    if rot is None:
        rot = [r["expected"] for r in rows[1:]] + [r["expected"] for r in rows[:1]]
    floor = [judge_L2(r["prediction"], t) for r, t in zip(rows, rot)]
    fp, flo, fhi = bootstrap_ci(floor)
    out["negative_control_floor"] = {"L2_floor": round(fp, 4),
                                     "ci": [round(flo, 4), round(fhi, 4)],
                                     "L2_margin": round(out["L2_contain_1sent"] - fp, 4)}
    strata = {}
    for tag in sorted({r["conflict_tag"] for r in rows}):
        sel = [i for i, r in enumerate(rows) if r["conflict_tag"] == tag]
        strata[tag] = {"n": len(sel),
                       "L2_contain_1sent": round(sum(L2[i] for i in sel) / len(sel), 4),
                       "L3_slot_exact": round(sum(L3[i] for i in sel) / len(sel), 4)}
    out["conflict_tag_strata"] = strata
    return out
"""

(OUT_DIR / "eval_judges.py").write_text(HARNESS, encoding="utf-8")
print("wrote", OUT_DIR / "eval_judges.py", len(HARNESS), "bytes")


wrote C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\eval_contract\artifacts\eval_judges.py 2788 bytes


In [16]:
summary = {
    "contract_version": "v1",
    "owner": "Fong Chin Wai (MC653748)",
    "generated": pd.Timestamp.now().isoformat(timespec="seconds"),
    "judge_levels": ["L0 field (legacy)", "L1 contain_anywhere", "L2 contain_first_sentence", "L3 slot_exact"],
    "benchmarks": {},
}
for label, df in FRAMES.items():
    bits = BITS[label]
    entry = {"n": len(df), "judge_rates": {}, "judge_ci": {}}
    for level in bits:
        p, lo, hi = bootstrap_ci(bits[level])
        entry["judge_rates"][level] = round(p, 4)
        entry["judge_ci"][level] = [round(lo, 4), round(hi, 4)]
    d = drift[drift["benchmark"] == label].iloc[0]
    entry["prefix_anchor_drift"] = {k: (int(d[k]) if k in ("both", "pos_only", "gen_only", "neither") else float(d[k]))
                                    for k in ("both", "pos_only", "gen_only", "neither", "phantom_rate", "mcnemar_p")}
    entry["negative_control"] = ctrl[ctrl["benchmark"] == label].set_index("judge")["floor"].to_dict()
    entry["strata_by_conflict_tag"] = strata_tag[strata_tag["benchmark"] == label].set_index("conflict_tag").to_dict("index")
    entry["strata_by_target_len"] = strata_len[strata_len["benchmark"] == label].set_index("target_len").to_dict("index")
    summary["benchmarks"][label] = entry

path = OUT_DIR / "eval_contract_summary.json"
path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", path)

for label, e in summary["benchmarks"].items():
    print(f"\n{label} (n={e['n']})")
    for k, v in e["judge_rates"].items():
        lo, hi = e["judge_ci"][k]
        print(f"   {k:24s} {v:.3f}  CI[{lo:.3f},{hi:.3f}]")


wrote C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\eval_contract\artifacts\eval_contract_summary.json

MQuAKE (n=50)
   L0 gen (as reported)     0.180  CI[0.080,0.300]
   L0 pos (as reported)     0.900  CI[0.820,0.980]
   L1 contain_anywhere      0.560  CI[0.420,0.700]
   L2 contain_1st_sent      0.560  CI[0.420,0.700]
   L3 slot_exact            0.060  CI[0.000,0.140]

WikiBigEdit (n=50)
   L0 gen (as reported)     0.820  CI[0.700,0.920]
   L0 pos (as reported)     0.800  CI[0.680,0.900]
   L1 contain_anywhere      0.720  CI[0.600,0.840]
   L2 contain_1st_sent      0.720  CI[0.600,0.840]
   L3 slot_exact            0.300  CI[0.180,0.420]


## 10 · What this changes

Everything below is printed from the numbers computed above — no hand-typed values.


In [17]:
lines = []
def say(s=""):
    lines.append(s)

say("## Diagnostics summary")
say()
for label, e in summary["benchmarks"].items():
    j = e["judge_rates"]; c = e["judge_ci"]
    d = e["prefix_anchor_drift"]
    say(f"### {label}  (n={e['n']})")
    say()
    say(f"- reported `gen_rewrite_hit` = {j['L0 gen (as reported)']:.2f}  CI[{c['L0 gen (as reported)'][0]:.2f},{c['L0 gen (as reported)'][1]:.2f}]")
    say(f"- reported `pos_rewrite_hit` = {j['L0 pos (as reported)']:.2f}  CI[{c['L0 pos (as reported)'][0]:.2f},{c['L0 pos (as reported)'][1]:.2f}]")
    say(f"- L1 contain_anywhere      = {j['L1 contain_anywhere']:.2f}")
    say(f"- **L2 contain_1st_sent    = {j['L2 contain_1st_sent']:.2f}**  CI[{c['L2 contain_1st_sent'][0]:.2f},{c['L2 contain_1st_sent'][1]:.2f}]  <- contract primary")
    say(f"- L3 slot_exact            = {j['L3 slot_exact']:.2f}")
    say(f"- prefix-anchor drift: pos_only={d['pos_only']}, gen_only={d['gen_only']}, McNemar p={d['mcnemar_p']:.3g}")
    say(f"- negative-control floor (L2) = {e['negative_control']['L2 contain_1st_sent']:.2f}  -> margin = {j['L2 contain_1st_sent'] - e['negative_control']['L2 contain_1st_sent']:+.2f}")
    pre = e["strata_by_conflict_tag"].get("pre-training-conflict")
    if pre:
        say(f"- pre-training-conflict: n={int(pre['n'])}, L2={pre['L2 contain_1st_sent']:.2f}, L3={pre['L3 slot_exact']:.2f}")
    say()

say("## Contract rules that take effect for all three of us")
say()
say("1. **Name the judge level on every number.** A bare `rewrite_acc` is no longer accepted.")
say("2. **L2 `contain_first_sentence` is the primary metric.** L1/L3 are reported alongside, never alone.")
say("3. **`pos_*` is a different measurement, not a headline.** Quote it only as prefix-anchored likelihood.")
say("4. **Report the negative-control floor and the margin.** Margin is what we compare across methods.")
say("5. **Every rate carries a bootstrap 95% CI.** n=50 gives roughly ±0.12; say so out loud.")
say("6. **Store the full generation** (drop the 120-char cap) plus `conflict_tag`.")
say("7. **Stratify by `conflict_tag`** in every results table.")
say("8. **Paired comparisons use McNemar on the same samples**, not CI overlap.")

report = "\n".join(lines)
print(report)
(OUT_DIR / "diagnostics_summary.md").write_text(report, encoding="utf-8")
print("\nwrote", OUT_DIR / "diagnostics_summary.md")


## Diagnostics summary

### MQuAKE  (n=50)

- reported `gen_rewrite_hit` = 0.18  CI[0.08,0.30]
- reported `pos_rewrite_hit` = 0.90  CI[0.82,0.98]
- L1 contain_anywhere      = 0.56
- **L2 contain_1st_sent    = 0.56**  CI[0.42,0.70]  <- contract primary
- L3 slot_exact            = 0.06
- prefix-anchor drift: pos_only=17, gen_only=0, McNemar p=1.53e-05
- negative-control floor (L2) = 0.00  -> margin = +0.56
- pre-training-conflict: n=5, L2=0.00, L3=0.00

### WikiBigEdit  (n=50)

- reported `gen_rewrite_hit` = 0.82  CI[0.70,0.92]
- reported `pos_rewrite_hit` = 0.80  CI[0.68,0.90]
- L1 contain_anywhere      = 0.72
- **L2 contain_1st_sent    = 0.72**  CI[0.60,0.84]  <- contract primary
- L3 slot_exact            = 0.30
- prefix-anchor drift: pos_only=4, gen_only=0, McNemar p=0.125
- negative-control floor (L2) = 0.08  -> margin = +0.64
- pre-training-conflict: n=9, L2=0.00, L3=0.00

## Contract rules that take effect for all three of us

1. **Name the judge level on every number.** A bare `

## 11 · Artifacts handed to the team

| file | who uses it | purpose |
|---|---|---|
| `artifacts/eval_judges.py` | Sonia, Tang | import `judge_L2` / `summarise` / `assert_contract`; stop re-implementing judges per script |
| `artifacts/eval_contract_summary.json` | all three | every rate + CI + floor + strata, machine-readable |
| `artifacts/diagnostics_summary.md` | supervisor / report | the quote-ready version of §10 |
| `artifacts/fig_judge_levels.png` | slides | the pos-vs-gen contrast in one picture |
| `artifacts/drift_cases_*.csv` | Vivi | the actual 17 + 4 discordant samples, readable one by one |

**Next for me (Vivi):** the writer swap. This notebook is the measuring stick the new writer gets
judged against — `MEMIT`/`AlphaEdit` closed-form results must be reported as
`L2 contain_1st_sent + margin`, stratified by `conflict_tag`, with the pre-training-conflict stratum
declared as a hard 0 unless it genuinely moves.
